# 01 · Convert the raw CSV to Parquet

**Goal:** turn the 16 GB `train_data.csv` into a compact Parquet file **once**, so every later notebook reads a few GB instead of 16.

**Before running:**
1. Add the competition data: *Add Input → Competitions → American Express - Default Prediction*.
2. Turn **Internet on** (Settings panel) so the notebook can clone the repo.
3. A CPU session is enough. No GPU needed here.

All real logic lives in `src/amex/data/convert.py`. This notebook only calls it and shows the results.

## 1. Get the project code

In [ ]:
REPO_URL = "https://github.com/<your-username>/amex-default-prediction.git"  # <- edit

import os, sys
if not os.path.exists("amex-default-prediction"):
    !git clone --depth 1 {REPO_URL}
sys.path.insert(0, "amex-default-prediction/src")

import polars as pl
print("polars", pl.__version__)

In [ ]:
from pathlib import Path

from amex.config import load_config
from amex.data.convert import check_id_encoding, convert_labels, convert_statements, summarize
from amex.utils.memory import file_gb, frame_gb, rss_gb, track

cfg = load_config("amex-default-prediction/configs/config.yaml")
raw_train, raw_labels = cfg["paths"]["raw_train"], cfg["paths"]["raw_labels"]
out_dir = Path(cfg["paths"]["processed_dir"])
train_pq, labels_pq = out_dir / "train.parquet", out_dir / "train_labels.parquet"

print(f"train CSV : {file_gb(raw_train):.2f} GB")
print(f"labels CSV: {file_gb(raw_labels):.3f} GB")
print(f"RAM used now: {rss_gb():.2f} GB")

## 2. Check that shortened customer IDs stay unique

We store each 64-character hex ID as a 60-bit integer (8 bytes instead of ~64). If two customers ended up with the same integer, their statements would be merged, which would silently corrupt the data. The labels file has one row per customer, so it is a cheap place to check.

In [ ]:
with track("ID uniqueness check"):
    check_id_encoding(raw_labels, cfg)
print("OK: no collisions")

## 3. Convert labels and statements

In [ ]:
with track("labels"):
    convert_labels(raw_labels, labels_pq, cfg)

# Streams the 16 GB CSV in batches. Expect several minutes.
# Watch the RAM gauge on the right: it should stay far below 16 GB.
with track("statements"):
    convert_statements(raw_train, train_pq, cfg)

## 4. Did it work? Sizes, counts, types

In [ ]:
s = summarize(train_pq, cfg)
s

In [ ]:
# Expected from the competition: ~5.53M statements, 458,913 customers, 190 columns
n_labels = pl.scan_parquet(labels_pq).select(pl.len()).collect().item()
assert s["customers"] == n_labels, "every customer in train should have a label"
print(f"customers: {s['customers']:,} | statements: {s['rows']:,}")

## 5. Memory: before vs after

These are the numbers for the README and the CV line *"memory-efficient Polars pipeline"*. Write them down.

In [ ]:
with track("load full parquet into RAM"):
    df = pl.read_parquet(train_pq)

print(f"CSV on disk      : {file_gb(raw_train):6.2f} GB")
print(f"Parquet on disk  : {file_gb(train_pq):6.2f} GB")
print(f"In RAM (Float32) : {frame_gb(df):6.2f} GB")
print(f"Same data as Float64 would be about {frame_gb(df) * 2:.1f} GB in RAM")
df.head()

In [ ]:
del df  # free RAM before the notebook finishes

## 6. Save the output for the next notebooks

1. Click **Save Version → Save & Run All**.
2. When it finishes, open the version's **Output** tab and click **New Dataset** (or, in later notebooks, *Add Input → Notebook Output* and select this notebook).

Later notebooks then read `train.parquet` directly and never touch the CSV again.